# 05 — NutriScan Evaluation

Evidence artefact for the complete project. Covers:

- **Section A** — Freshness model evaluation (MAE, RMSE, accuracy)
- **Section B** — Portion estimation spot check
- **Section C** — Agent trace (offline scoring)
- **Section D** — Summary table (resume bullets)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROCESSED = Path("data/processed")
CHECKPOINT = PROCESSED / "freshness_best.pt"

print(f"Checkpoint exists: {CHECKPOINT.exists()}")
if not CHECKPOINT.exists():
    raise SystemExit(
        "No checkpoint found. Train the model first:\n"
        "  uv run python -m data.download\n"
        "  uv run python -m models.freshness.preprocess\n"
        "  uv run python -m models.freshness.train"
    )

---
## Section A — Freshness Model Evaluation

In [ ]:
from models.freshness.model import FreshnessRegressor

# Load test split
emb_path = PROCESSED / "embeddings_test.pt"
lab_path = PROCESSED / "labels_test.pt"
X_test = torch.load(emb_path, weights_only=True)
y_test = torch.load(lab_path, weights_only=True)

print(f"Test samples: {X_test.shape[0]}")
print(f"Embedding dim: {X_test.shape[1]}")
print(f"Label range: [{y_test.min():.2f}, {y_test.max():.2f}]")

In [ ]:
# Load model from checkpoint
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=True)

model = FreshnessRegressor(
    input_dim=ckpt["config"]["input_dim"],
    device=device,
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
print(f"Val loss at save: {ckpt['val_loss']:.6f}")

In [ ]:
# Point predictions (no MC dropout)
with torch.no_grad():
    preds_point = model(X_test.to(device)).cpu().squeeze()

# MC Dropout predictions
mc_mean, mc_std = model.predict_with_uncertainty(X_test.to(device), n_passes=20)
preds_mc = mc_mean.cpu().squeeze()
uncertainty = mc_std.cpu().squeeze()

y_np = y_test.numpy()
pred_np = preds_point.numpy()
mc_np = preds_mc.numpy()
unc_np = uncertainty.numpy()

In [ ]:
# Compute metrics
mae = float(np.mean(np.abs(pred_np - y_np)))
rmse = float(np.sqrt(np.mean((pred_np - y_np) ** 2)))

pred_labels = (pred_np >= 0.5).astype(int)
true_labels = (y_np >= 0.5).astype(int)
accuracy = float(np.mean(pred_labels == true_labels))

baseline_mae = float(np.mean(np.abs(0.5 - y_np)))
improvement = (baseline_mae - mae) / baseline_mae * 100

print(f"{'Test MAE':<30} {mae:.4f}")
print(f"{'Test RMSE':<30} {rmse:.4f}")
print(f"{'Accuracy @ 0.5':<30} {accuracy:.4f}")
print(f"{'Naive baseline MAE':<30} {baseline_mae:.4f}")
print(f"{'Improvement over baseline':<30} {improvement:.1f}%")

In [ ]:
# Plot 1: Histogram of predicted scores by true label
fig, ax = plt.subplots(figsize=(8, 5))

fresh_mask = y_np >= 0.5
ax.hist(
    pred_np[fresh_mask],
    bins=30,
    alpha=0.6,
    label="Fresh (true)",
    color="#2ecc71",
)
ax.hist(
    pred_np[~fresh_mask],
    bins=30,
    alpha=0.6,
    label="Rotten (true)",
    color="#e74c3c",
)
ax.axvline(
    x=0.5,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Threshold (0.5)",
)

ax.set_xlabel("Predicted Freshness Score")
ax.set_ylabel("Count")
ax.set_title("Distribution of Predicted Freshness Scores")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Score vs uncertainty, coloured by correctness
mc_labels = (mc_np >= 0.5).astype(int)
correct = mc_labels == true_labels

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    mc_np[correct],
    unc_np[correct],
    alpha=0.5,
    s=20,
    label="Correct",
    color="#2ecc71",
)
ax.scatter(
    mc_np[~correct],
    unc_np[~correct],
    alpha=0.7,
    s=30,
    label="Incorrect",
    color="#e74c3c",
    marker="x",
)

ax.set_xlabel("Predicted Freshness Score (MC mean)")
ax.set_ylabel("Uncertainty (MC std)")
ax.set_title("Prediction Confidence vs Uncertainty")
ax.legend()
plt.tight_layout()
plt.show()

---
## Section B — Portion Estimation Spot Check

> **Note:** Ground-truth gram weights are not available in the
> dataset. This section is a qualitative check. The geometric
> heuristic is a valid first-order approximation because:
> 1. Food items have consistent density per category.
> 2. A reference object (plate) provides pixel-to-cm² calibration.
> 3. Bounding box area × typical height → volume proxy.

In [ ]:
from models.portion.estimator import PortionEstimator

raw_dir = Path("data/raw")
jpgs = sorted(raw_dir.rglob("*.jpg"))[:5]
pngs = sorted(raw_dir.rglob("*.png"))[:5]
images = (jpgs + pngs)[:5]

if not images:
    print("No images in data/raw/ — skipping.")
else:
    estimator = PortionEstimator()
    hdr = f"{'Image':<40} {'Label':<12}"
    hdr += f" {'Grams':>8} {'±':>8} {'Conf':>6}"
    print(hdr)
    print("-" * 80)
    for img_path in images:
        results = estimator.estimate(str(img_path))
        if not results:
            none_label = "(none)"
            print(f"{img_path.name:<40} {none_label}")
        for r in results:
            print(
                f"{img_path.name:<40} "
                f"{r.label:<12} "
                f"{r.estimated_grams:>8.1f} "
                f"{r.uncertainty_grams:>8.1f} "
                f"{r.detection_confidence:>6.3f}"
            )

---
## Section C — Agent Trace

Runs the scoring pipeline offline (no MongoDB).

**Scoring formula:**
```
score = 0.7 × macro_fit + 0.3 × freshness_bonus
```
- `macro_fit = 1 − mean(|recipe − deficit| / max(deficit, 1))`
- `freshness_bonus = mean freshness of matching fridge items`

In [ ]:
from datetime import UTC, datetime

from agent.tools import (
    compute_macro_deficit,
    load_recipes,
    score_recipe,
)
from db.models import (
    BoundingBox,
    DetectedItem,
    FridgeState,
    MacroTargets,
    UserProfile,
)

# Construct in-memory state
profile = UserProfile(
    user_id="eval_user",
    display_name="Eval User",
    daily_targets=MacroTargets(
        calories=2200,
        protein_g=60,
        carbs_g=275,
        fat_g=73,
    ),
    dietary_restrictions=["vegetarian"],
)

bb1 = BoundingBox(x_min=0, y_min=0, x_max=100, y_max=100)
bb2 = BoundingBox(x_min=0, y_min=0, x_max=80, y_max=120)
bb3 = BoundingBox(x_min=0, y_min=0, x_max=60, y_max=60)

fridge = FridgeState(
    user_id="eval_user",
    image_path="synthetic",
    captured_at=datetime.now(tz=UTC),
    detected_items=[
        DetectedItem(
            name="apple",
            bounding_box=bb1,
            confidence=0.9,
            freshness_score=0.85,
            estimated_grams=180,
        ),
        DetectedItem(
            name="banana",
            bounding_box=bb2,
            confidence=0.88,
            freshness_score=0.7,
            estimated_grams=120,
        ),
        DetectedItem(
            name="tomato",
            bounding_box=bb3,
            confidence=0.82,
            freshness_score=0.92,
            estimated_grams=150,
        ),
    ],
)

deficit = compute_macro_deficit(profile.daily_targets, [])
print(f"Full-day deficit: {deficit}")

In [ ]:
recipes = load_recipes()
scored = []
for r in recipes:
    s = score_recipe(r, deficit, fridge)
    macros = r.get("macros", {})
    pairs = [
        (macros.get("calories", 0), deficit.calories),
        (macros.get("protein_g", 0), deficit.protein_g),
        (macros.get("carbs_g", 0), deficit.carbs_g),
        (macros.get("fat_g", 0), deficit.fat_g),
    ]
    errs = [abs(rv - tv) / max(tv, 1.0) for rv, tv in pairs]
    mfit = max(0.0, min(1.0, 1 - sum(errs) / len(errs)))

    uses = {n.lower() for n in r.get("uses_ingredients", [])}
    fn = {
        it.name.lower(): it.freshness_score
        for it in fridge.detected_items
        if it.freshness_score is not None
    }
    matched = [fn[n] for n in uses if n in fn]
    fb = sum(matched) / len(matched) if matched else 0.0

    scored.append(
        {
            "name": r["name"],
            "macro_fit": mfit,
            "freshness_bonus": fb,
            "score": s,
        }
    )

scored.sort(key=lambda x: x["score"], reverse=True)

hdr = f"{'Rank':<6} {'Recipe':<30}"
hdr += f" {'Macro Fit':>10} {'Fresh':>8} {'Score':>8}"
print(hdr)
print("-" * 66)
for i, r in enumerate(scored[:5], 1):
    print(
        f"{i:<6} {r['name']:<30}"
        f" {r['macro_fit']:>10.4f}"
        f" {r['freshness_bonus']:>8.4f}"
        f" {r['score']:>8.4f}"
    )

---
## Section D — Summary Table

| Bullet | Implementation | Key Metric |
|--------|---------------|------------|
| Freshness regression | CLIP ViT-B/32 + MLP | Test MAE: see A |
| Portion estimation | YOLOv8n + geometric proxy | Qualitative |
| LangGraph agent | 7-node state machine | 30 recipes |

In [ ]:
# Print final summary with real values
print("=" * 60)
print("NUTRISCAN  EVALUATION SUMMARY")
print("=" * 60)
print(f"Freshness MAE:       {mae:.4f}")
print(f"Freshness RMSE:      {rmse:.4f}")
print(f"Accuracy @ 0.5:      {accuracy:.4f}")
print(f"Baseline MAE:        {baseline_mae:.4f}")
print(f"Improvement:         {improvement:.1f}%")
print(f"Recipe corpus:       {len(recipes)}")
top = scored[0]
print(f"Top recipe:          {top['score']:.4f} ({top['name']})")
print("=" * 60)